### Query Data from MassWiki

In [1]:
import pandas as pd
import requests
from urllib.parse import quote
from typing import List, Dict, Optional, Any, Tuple

def load_masswiki_results() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load mass spectrometry results from CSV files."""
    ttof_result = pd.read_csv("data/hilic_ttof_neg.csv")
    orb_result = pd.read_csv("data/hilic_pre_orb_neg.csv")
    return ttof_result, orb_result

def filter_masswiki_results(masswiki_result: pd.DataFrame) -> pd.DataFrame:
    """
    Filter mass wiki results to remove entries without user annotations
    and entries with names starting with 'yy' or 'zz'.
    """
    filtered_result = masswiki_result[
        (masswiki_result['is_manual_annotated'] == True) &
        (~masswiki_result['name'].str.match(r'^(yy|zz)', na=False))
    ]
    return filtered_result

def get_spectrum_data(wiki_ids: List[str]) -> Dict[str, Dict]:
    """
    Fetch spectrum data from MassWiki API for given wiki IDs.
    
    Args:
        wiki_ids: List of wiki IDs to query
        
    Returns:
        Dictionary mapping wiki IDs to their spectrum data
    """
    all_results = {}
    
    for wiki_id in wiki_ids:
        # Skip empty or NA wiki_ids
        if pd.isna(wiki_id) or wiki_id == "":
            print(f"Skipping empty or NA wiki_id")
            continue
            
        # URL encode the wiki_id to handle special characters
        encoded_wiki_id = quote(wiki_id)
        url = f'https://masswiki.us-west-2.elasticbeanstalk.com/analysis/get_data?wiki_id={encoded_wiki_id}'
        
        try:
            response = requests.get(url, headers={'Accept': 'application/json'})
            
            if response.status_code == 200:
                data = response.json()
                results = {}
                
                # Extract reference library identity search results
                analysis_data = data.get('analysis', {})
                ref_lib_data = analysis_data.get('reference_library', {})
                results['reference_library'] = ref_lib_data.get('identity_search')
                
                # Extract annotation library identity search results
                annot_lib_data = analysis_data.get('annotation_library', {})
                results['annotation_library'] = annot_lib_data.get('identity_search')
                
                all_results[wiki_id] = results
            else:
                print(f"Failed to get spectrum data for {wiki_id}. Status code: {response.status_code}")
                all_results[wiki_id] = None
                
        except Exception as e:
            print(f"Error processing wiki_id: {wiki_id} - {str(e)}")
            all_results[wiki_id] = None
            
    return all_results

def main():
    # Load data
    ttof_result, orb_result = load_masswiki_results()
    
    # Filter results
    filtered_ttof_result = filter_masswiki_results(ttof_result)
    filtered_orb_result = filter_masswiki_results(orb_result)
    
    # Query spectrum data
    query_orb = get_spectrum_data(filtered_orb_result['wiki_id'].tolist())
    query_ttof = get_spectrum_data(filtered_ttof_result['wiki_id'].tolist())
    
    return query_orb, query_ttof

if __name__ == "__main__":
    query_orb, query_ttof = main()